In [1]:
import os
print(os.environ.get('LD_LIBRARY_PATH'))
print(os.environ.get('LD_PRELOAD'))

/tools/Xilinx/Vitis_HLS/2022.2/lib/lnx64.o:/tools/Xilinx/Vitis_HLS/2022.2/lnx64/lib/csim:/tools/Xilinx/Vitis_HLS/2022.2/tps/lnx64/gcc-8.3.0/lib64:
None


In [2]:
import os
os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libstdc++.so.6"
print(os.environ.get('LD_PRELOAD'))

/usr/lib/x86_64-linux-gnu/libstdc++.so.6


In [3]:
import os
os.environ['PATH'] = '/home/fitba/tools/Xilinx/Vitis_HLS/2022.2/bin:' + os.environ['PATH'] 
import tensorflow as tf
import hls4ml
import numpy as np

# =====================================================================
# 1. BLOQUE DE PARAMETRIZACIÓN GLOBAL (Link con Etapa 4)
# =====================================================================
# A. Parámetros de Entrenamiento (Origen)
METODO_COMPRESION = "Destillation"
EPOCHS_TRAIN      = 15
BATCH_SIZE_TRAIN  = 128
NOMBRE_MODELO     = "modelo_estudiante_destilado_qat"
BITS_TOTALES      = '8'
BITS_PARTE_ENTERA = '2'

# B. Rutas Dinámicas
DIR_MODELOS   = f"/mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/models/{METODO_COMPRESION}_{EPOCHS_TRAIN}epochs_{BATCH_SIZE_TRAIN}batch"
RUTA_MODELO   = f"{DIR_MODELOS}/{NOMBRE_MODELO}.h5"
DIR_HLS_OUT   = f"hls_Prj/pynq_{NOMBRE_MODELO}_prj"

# C. Parámetros de Arquitectura de Hardware (PYNQ-Z2)
TARGET_BOARD  = 'pynq-z2'
TARGET_PART   = 'xc7z020clg400-1'
IO_PROTOCOL   = 'io_stream' # Requisito para AXI-DMA

# D. Parámetros de Síntesis Matemática (Optimizables según recursos)
PRECISION_HW  = f'ap_fixed<{BITS_TOTALES},{BITS_PARTE_ENTERA}>'
FACTOR_REUSO  = 64                # 1 = Totalmente paralelo (Baja latencia). Subir a 2 o 4 si faltan DSPs.
ESTRATEGIA    = 'Resource'        # Priorizar velocidad ('Latency') vs área ('Resource')

print(f"--- Iniciando Pipeline HLS ---")
print(f"Cargando modelo desde: {RUTA_MODELO}")
print(f"Directorio de salida : {DIR_HLS_OUT}")

2026-06-24 18:57:13.852369: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-24 18:57:13.853606: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-24 18:57:13.872120: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-24 18:57:13.872520: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-24 18:57:14.237362: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

--- Iniciando Pipeline HLS ---
Cargando modelo desde: /mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/models/Destillation_15epochs_128batch/modelo_estudiante_destilado_qat.h5
Directorio de salida : hls_Prj/pynq_modelo_estudiante_destilado_qat_prj


In [4]:
# =====================================================================
# 2. CARGA DEL MODELO DESTILADO (Corregido para QKeras)
# =====================================================================
from qkeras.utils import load_qmodel

print(f"Cargando modelo cuantizado desde: {RUTA_MODELO}...")

# Keras puro fallaría aquí. Usamos load_qmodel de QKeras que ya conoce 
# toda la topología de hardware (QDense, QActivation, etc.)
modelo_keras = load_qmodel(RUTA_MODELO, compile=False)

modelo_keras.summary()

Cargando modelo cuantizado desde: /mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/models/Destillation_15epochs_128batch/modelo_estudiante_destilado_qat.h5...
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 qdense_1 (QDense)           (None, 32)                608       
                                                                 
 batch_normalization (BatchN  (None, 32)               128       
 ormalization)                                                   
                                                                 
 q_activation (QActivation)  (None, 32)                0         
                                                                 
 qdense_2 (QDense)           (None, 16)                528       
                                                                 
 batch_normalization_1 (Batc  (None, 16)               64        
 hNormalization)                        

In [5]:
# =====================================================================
# 3. CONFIGURACIÓN DEL PERFILADOR HLS4ML
# =====================================================================
config_hls = hls4ml.utils.config_from_keras_model(modelo_keras, granularity='model')

# Aplicamos los parámetros dinámicos de hardware
config_hls['Model']['Precision'] = PRECISION_HW
config_hls['Model']['ReuseFactor'] = FACTOR_REUSO
config_hls['Model']['Strategy'] = ESTRATEGIA

print("\n--- Conf>iguración Matemática Generada ---")
print(config_hls)


Interpreting Sequential
Topology:
Layer name: qdense_1_input, layer type: InputLayer, input shapes: [[None, 18]], output shape: [None, 18]
Layer name: qdense_1, layer type: QDense, input shapes: [[None, 18]], output shape: [None, 32]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 32]], output shape: [None, 32]
Layer name: q_activation, layer type: Activation, input shapes: [[None, 32]], output shape: [None, 32]
Layer name: qdense_2, layer type: QDense, input shapes: [[None, 32]], output shape: [None, 16]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: q_activation_1, layer type: Activation, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: output, layer type: QDense, input shapes: [[None, 16]], output shape: [None, 1]

--- Conf>iguración Matemática Generada ---
{'Model': {'Precision': 'ap_fixed<8,2>', 'ReuseFactor': 64, 'Strategy': 'Resource', 'Bram

In [6]:
# =====================================================================
# 4. CONVERSIÓN Y COMPILACIÓN (C-SIMULATION)
# =====================================================================
import shutil # Agrega esto junto a tus otros imports arriba (os, tf, hls4ml)

# =====================================================================
# 4. CONVERSIÓN Y COMPILACIÓN (C-SIMULATION)
# =====================================================================
print("\n[1/3] Preparando entorno y convirtiendo modelo Keras a C++...")

if os.path.exists(DIR_HLS_OUT):
    print(f"      -> Detectada compilación previa. Limpiando directorio: {DIR_HLS_OUT}")
    shutil.rmtree(DIR_HLS_OUT)
# ---------------------------------------------------------------------

hls_model = hls4ml.converters.convert_from_keras_model(
    modelo_keras,
    hls_config=config_hls,
    output_dir=DIR_HLS_OUT,
    part=TARGET_PART,
    # board=TARGET_BOARD,
    backend='Vitis',
    io_type=IO_PROTOCOL
)

print("[2/3] Compilando el modelo en C++...")
hls_model.compile()

# Opcional: Aquí podrías hacer predicciones de prueba con datos
# y_hls = hls_model.predict(X_test_param)


[1/3] Preparando entorno y convirtiendo modelo Keras a C++...
Interpreting Sequential
Topology:
Layer name: qdense_1_input, layer type: InputLayer, input shapes: [[None, 18]], output shape: [None, 18]
Layer name: qdense_1, layer type: QDense, input shapes: [[None, 18]], output shape: [None, 32]
Layer name: batch_normalization, layer type: BatchNormalization, input shapes: [[None, 32]], output shape: [None, 32]
Layer name: q_activation, layer type: Activation, input shapes: [[None, 32]], output shape: [None, 32]
Layer name: qdense_2, layer type: QDense, input shapes: [[None, 32]], output shape: [None, 16]
Layer name: batch_normalization_1, layer type: BatchNormalization, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: q_activation_1, layer type: Activation, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: output, layer type: QDense, input shapes: [[None, 16]], output shape: [None, 1]
Creating HLS model
[2/3] Compilando el modelo en C++...
Writing HLS pr

Done


In [7]:
# =====================================================================
# 5. SÍNTESIS DE HARDWARE Y GENERACIÓN DE BITSTREAM
# =====================================================================
# ATENCIÓN: Esta celda ejecutará Vivado en segundo plano. Puede demorar varios minutos.
print("[3/3] Iniciando Síntesis HLS y Generación de Bitfile...")

# csim=False: omite la simulación en C.
# synth=True: traduce C++ a RTL (VHDL/Verilog).
# export=True: empaqueta el RTL en un IP Core (.zip) estándar de Xilinx.
compilation = hls_model.build(
    csim=False, 
    synth=True, 
    export=True
)

if compilation:
    print(f"\n--- ¡Síntesis HLS Completada Exitosamente! ---")
    print(f"Tu modelo ha sido empaquetado como un IP Core. Búscalo en:")
    print(f"-> {DIR_HLS_OUT}/myproject_prj/solution1/impl/ip/")

[3/3] Iniciando Síntesis HLS y Generación de Bitfile...

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2022.2 (64-bit)
  **** SW Build 3670227 on Oct 13 2022
  **** IP Build 3669848 on Fri Oct 14 08:30:02 MDT 2022
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.

source /home/fitba/tools/Xilinx/Vitis_HLS/2022.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/home/fitba/tools/Xilinx/Vitis_HLS/2022.2/bin/unwrapped/lnx64.o/vitis_hls'
INFO: [HLS 200-10] For user 'fitba' on host 'seaman-fitba2' (Linux_x86_64 version 6.12.86+deb13-amd64) on Wed Jun 24 18:57:19 -03 2026
INFO: [HLS 200-10] On os Debian GNU/Linux 13 (trixie)
INFO: [HLS 200-10] In directory '/mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/notebooks/Etapa_5/hls_Prj/pynq_modelo_estudiante_destilado_qat_prj'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Creating and opening project '/mnt/Datos/Julian_Alvarez-TFG/IDS-IOT/notebooks/

In [8]:
hls4ml.report.read_vivado_report(f'{DIR_HLS_OUT}/')

Found 1 solution(s) in hls_Prj/pynq_modelo_estudiante_destilado_qat_prj//myproject_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vitis HLS Report for 'myproject'
* Date:           Wed Jun 24 18:57:41 2026

* Version:        2022.2 (Build 3670227 on Oct 13 2022)
* Project:        myproject_prj
* Solution:       solution1 (Vivado IP Flow Target)
* Product family: zynq
* Target device:  xc7z020-clg400-1


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+---------+----------+------------+
    |  Clock |  Target | Estimated| Uncertainty|
    +--------+---------+----------+------------+
    |ap_clk  |  5.00 ns|  7.898 ns|     0.62 ns|
    +--------+---------+----------+------------+

+ Latency: 
    * Summary: 
    +---------+---------+----------+----------+-----+-----+----------+
    |  Latency (cycles) |  Latency (absolute) |  Interval | Pipeline |
    |   min   |   max   |    min   |    max   | min | max |   Type   |
    +----